In [ ]:
# GPU check
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.randn(100, 100, device="cuda")
    print("CUDA computation:", x.sum().item())

# stAI

To simplify running stAI programmatically, we modified the package so that configuration parameters can be passed directly in the code, rather than being loaded from an external YAML file.

In [ ]:
import spatialdata as sd
import scanpy as sc
import os
import pandas as pd
import stAI
import shutil
from Xenium_5K_IPF_constants import IMP_GENE_NAMES, SAMPLES_SYMBOLS, WORK_PATH, PUBLIC_PATH

# figure directory
os.makedirs(f"{WORK_PATH}Imputation/stAI", exist_ok=True)
os.makedirs(f"{WORK_PATH}Imputation/stAI/csv", exist_ok=True)

# Option
sc.settings.verbosity = 3

In [ ]:
sub_type_list = [
    "Epithelial", 
    "Macrophage", 
    "Fibroblast"
]
GSE_list = [
    "135893",
    "136831"
]
overwrite = True  # If you use data that has already been calculated using stAI, set overwrite to False.

In [ ]:
model_parameters_imp={
    'd_hidden': 64,
    'd_latent': 16,
    'lam_clf': 0.1,
    'lam_cos': 1,
    'lam_genegraph': 0,
    'lam_impute': 1,
    'lam_mmd': 0.1,
    'lam_recon': 1
}

training_parameters_imp = {
    'device_id': 0,
    'eval_step': 50,
    'k_folds': 5,
    'lr': 0.002,
    'num_epoch': 200,
    'rna_batchsize': 1024,
    'spatial_batchsize': 1024,
    'spatial_knn': 10
}

model_parameters_unm={
    'd_hidden': 64,
    'd_latent': 16,
    'lam_clf': 0.1,
    'lam_cos': 1,
    'lam_genegraph': 0,
    'lam_impute': 1,
    'lam_mmd': 0.1,
    'lam_recon': 1
}
training_parameters_unm={
    "device_id": 0,
    "k_folds": 5,
    "spatial_knn": 10
}

In [ ]:
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/011_1st_clustering.csv", index_col="cell_id")

In [ ]:
for GSE in GSE_list:
    os.makedirs(f"{WORK_PATH}Imputation/stAI/GSE{GSE}", exist_ok=True)
    os.makedirs(f"{WORK_PATH}Imputation/stAI/csv/GSE{GSE}", exist_ok=True)
    
    for sub_type in sub_type_list:
        ref_adata = sc.read_h5ad(f"{PUBLIC_PATH}GSE{GSE}/ws/filtered_data.h5ad")   # filtered IPF and Control groups
        if sub_type == "Epithelial":    
            if GSE =="136831":
                adata = ref_adata[ref_adata.obs["CellType_Category"]=='Epithelial'].copy()
                adata.obs=adata.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                adata = adata[adata.obs["celltype"]!='ATII_Low-Surfactants_SubjSpec']
                adata.obs["celltype"] = adata.obs["celltype"].cat.remove_unused_categories()
            else:  # 135893
                adata = ref_adata[ref_adata.obs["population"]=="Epithelial"].copy()
        
        elif sub_type == "Fibroblast":    
            if GSE =="136831":
                adata = ref_adata[ref_adata.obs["Subclass_Cell_Identity"].isin(["Fibroblast", 'Myofibroblast'])].copy()
                adata.obs=adata.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                adata.obs["celltype"] = adata.obs["celltype"].cat.remove_unused_categories()
            else:  # 135893
                adata = ref_adata[ref_adata.obs["celltype"].isin(['PLIN2+ Fibroblasts', 'Fibroblasts', 'HAS1 High Fibroblasts', 'Myofibroblasts', 'Mesothelial Cells'])].copy()
        
        elif sub_type == "Macrophage":
            if GSE =="136831":
                gse_m = pd.read_table(f"{PUBLIC_PATH}GSE{GSE}/metadata.txt.gz")
                obs_df = pd.merge(left=ref_adata.obs, right=gse_m[["CellBarcode_Identity","Manuscript_Identity"]], left_index=True, right_on="CellBarcode_Identity", how="left")
                obs_df.index = obs_df["CellBarcode_Identity"]
                ref_adata.obs = obs_df[["Manuscript_Identity"]]
                adata = ref_adata[ref_adata.obs["Manuscript_Identity"].isin(['ncMonocyte', 'Macrophage_Alveolar','cMonocyte', 'Macrophage'])].copy()
                adata.obs=adata.obs.rename(columns={'Manuscript_Identity':"celltype"})
            else:  # 135893
                adata = ref_adata[ref_adata.obs["celltype"].isin(['Monocytes', 'Macrophages'])].copy()
                
        del ref_adata
        
        for i, ss in enumerate(SAMPLES_SYMBOLS):
            sp_adata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{ss}")
            df = celltype_df[celltype_df.index.str.endswith(str(i))]
            df.index = [n[:-1]+"1" for n in df.index]

            sp_adata_m = sp_adata.tables["table"]
            sp_adata_m = sp_adata_m[sp_adata_m.obs["cell_id"].isin(df.index.to_list())]
            sp_adata_m.obs = pd.merge(sp_adata_m.obs["cell_id"], df, left_on="cell_id", right_index=True) 
            sp_adata_m = sp_adata_m[sp_adata_m.obs["cell_type"]==sub_type]
        
            save_dir =f"{WORK_PATH}Imputation/stAI/GSE{GSE}/imputation_{sub_type.replace("/", "_")}_{ss}"


            if overwrite:
                if os.path.isdir(save_dir):
                    shutil.rmtree(save_dir)
                    print("Deleted old imputation data")
                print("Creat new imputation data")
                stAI.main.run_impute(
                    model_parameters=model_parameters_imp, 
                    training_parameters=training_parameters_imp,
                    adata_spatial=sp_adata_m, 
                    adata_rna=adata, 
                    save_dir=save_dir
                )
            else:
                assert os.path.isdir(save_dir), "Save directory does not exist"
                print("Use imputation data")
        
            res = stAI.main.impute_unmeasured_genes(
                genes_to_impute=IMP_GENE_NAMES[sub_type],
                adata_spatial=sp_adata_m.copy(),
                adata_rna=adata.copy(),
                training_parameters=training_parameters_unm,
                model_parameters=model_parameters_unm,
                model_dir=save_dir, 
                agg=True, 
                device="cpu"
            )
            
            df_res=pd.DataFrame(res, index=sp_adata_m.obs["cell_id"], columns=IMP_GENE_NAMES[sub_type])
                
            df_res.to_csv(f"{WORK_PATH}Imputation/stAI/csv/GSE{GSE}/{sub_type.replace("/", "_")}_imp_{ss}.csv", encoding="utf-8-sig")


# TANGRAM

In [ ]:
import spatialdata as sd
import scanpy as sc
import os
import pandas as pd
import numpy as np
import tangram as tg
import gc
import torch
from Xenium_5K_IPF_constants import SAMPLES_SYMBOLS, WORK_PATH, PUBLIC_PATH

# figure directory
os.makedirs(f"{WORK_PATH}Imputation/Tangram", exist_ok=True)

# Option
sc.settings.verbosity = 3

In [ ]:
sub_type_list = [
    "Epithelial", 
    "Macrophage", 
    "Fibroblast"
]
GSE_list = [
    "135893",
    "136831"
]

In [ ]:
celltype_df = pd.read_csv(f"{WORK_PATH}CSV/011_1st_clustering.csv")
for GSE in GSE_list:
    os.makedirs(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}", exist_ok=True)
    ref_adata = sc.read_h5ad(f"{PUBLIC_PATH}GSE{GSE}/ws/filtered_data.h5ad")  # filtered IPF and Control groups
    
    if GSE == "136831":
        gse_m = pd.read_table(f"{PUBLIC_PATH}GSE{GSE}/metadata.txt.gz", index_col="CellBarcode_Identity")
        ref_adata.obs = pd.merge(ref_adata.obs, gse_m[["Manuscript_Identity"]], left_index=True, right_index=True, how="left")
    
    for sub_type in sub_type_list:
        for n, ss in enumerate(SAMPLES_SYMBOLS):
            sp_adata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{ss}")
            
            df = celltype_df[celltype_df["cell_id"].str.endswith(str(n))].copy()
            df["cell_id"] = [d[:-1]+"1" for d in df["cell_id"]]
            df.index=df["cell_id"]
            
            ad_sp = sp_adata.tables["table"]
            ad_sp = ad_sp[ad_sp.obs["cell_id"].isin(df.index.to_list())].copy()
            ad_sp.obs = pd.merge(ad_sp.obs["cell_id"],df, left_on="cell_id", right_index=True) 
            
            ad_sp = ad_sp[ad_sp.obs["cell_type"]==sub_type].copy()
                
            if sub_type == "Epithelial":    
                if GSE =="136831":
                    ad_sc = ref_adata[ref_adata.obs["CellType_Category"]=='Epithelial'].copy()
                    ad_sc.obs=ad_sc.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                    ad_sc = ad_sc[ad_sc.obs["celltype"]!='ATII_Low-Surfactants_SubjSpec']
                    ad_sc.obs["celltype"] = ad_sc.obs["celltype"].cat.remove_unused_categories()
                else:  # 135893
                    ad_sc = ref_adata[ref_adata.obs["population"]=="Epithelial"].copy()
            
            elif sub_type == "Fibroblast":    
                if GSE =="136831":
                    ad_sc = ref_adata[ref_adata.obs["Subclass_Cell_Identity"].isin(["Fibroblast", 'Myofibroblast'])].copy()
                    ad_sc.obs=ad_sc.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                    ad_sc.obs["celltype"] = ad_sc.obs["celltype"].cat.remove_unused_categories()
                else:  # 135893
                    ad_sc = ref_adata[ref_adata.obs["celltype"].isin(['PLIN2+ Fibroblasts', 'Fibroblasts', 'HAS1 High Fibroblasts', 'Myofibroblasts', 'Mesothelial Cells'])].copy()
            
            elif sub_type == "Macrophage":
                if GSE =="136831":
                    ad_sc = ref_adata[ref_adata.obs["Manuscript_Identity"].isin(['ncMonocyte', 'Macrophage_Alveolar','cMonocyte', 'Macrophage'])].copy()
                    ad_sc.obs=ad_sc.obs.rename(columns={'Manuscript_Identity':"celltype"})
    
                    # Sampling
                    rng = np.random.default_rng(0)
                    keep_pos = []
                    for ct, pos in ad_sc.obs.groupby("celltype").indices.items():
                        pos = np.asarray(pos)
                        k = len(pos) // 4
                        keep_pos.append(rng.choice(pos, size=k, replace=False))
                    keep_pos = np.concatenate(keep_pos) if keep_pos else np.array([], dtype=int)
                    ad_sc = ad_sc[keep_pos,].copy()
                else:  # 135893
                    ad_sc = ref_adata[ref_adata.obs["celltype"].isin(['Monocytes', 'Macrophages'])].copy()
                
            sc.pp.normalize_per_cell(ad_sp, counts_per_cell_after=1e4)
            sc.pp.log1p(ad_sp)
            sc.pp.highly_variable_genes(ad_sp, n_top_genes=2000)
            sc.pp.normalize_per_cell(ad_sc, counts_per_cell_after=1e4)
            sc.pp.log1p(ad_sc)
            sc.pp.highly_variable_genes(ad_sc, n_top_genes=4000)
            sc_ge_list = ad_sc.var[ad_sc.var["highly_variable"]].index.tolist()
            sp_ge_list = ad_sp.var[ad_sp.var["highly_variable"]].index.tolist()
            ge_list = list(set(sc_ge_list) & set(sp_ge_list))
            
            tg.pp_adatas(ad_sc, ad_sp, genes=ge_list)
            
            ad_map = tg.map_cells_to_space(ad_sc, ad_sp, device='cuda:0')
            
            ad_ge = tg.project_genes(ad_map, ad_sc)
            
            ad_ge.write(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}/{ss}_{sub_type.replace("/", "_")}.h5ad")
            
            # GPU memory release
            del sp_adata, ad_sp, ad_sc, ad_map, ad_ge, df, ge_list, sc_ge_list, sp_ge_list
            gc.collect()
            torch.cuda.empty_cache()        

        adata_tangram_list = [sc.read_h5ad(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}/{ss}_{sub_type.replace("/", "_")}.h5ad") for ss in SAMPLES_SYMBOLS]
            
        for n in range(len(adata_tangram_list)):
            adata_tangram_list[n].obs_names=[adata_tangram_list[n].obs["cell_id"].iloc[m][:-2]+"-"+str(n) for m in range(len(adata_tangram_list[n].obs_names))]
        adata_tangram = adata_tangram_list[0].concatenate(adata_tangram_list[1:6])
        
        adata_tangram.write(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}/Tangram_{sub_type.replace("/", "_")}.h5ad")
        

# Validation

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
import anndata as ad
import matplotlib as mpl
import stAI
from scipy.stats import spearmanr, pearsonr
from scipy import sparse
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable

from Xenium_5K_IPF_constants import WORK_PATH, PUBLIC_PATH, SAMPLES_SYMBOLS

%matplotlib ipympl

In [ ]:
model_parameters_imp={
    'd_hidden': 64,
    'd_latent': 16,
    'lam_clf': 0.1,
    'lam_cos': 1,
    'lam_genegraph': 0,
    'lam_impute': 1,
    'lam_mmd': 0.1,
    'lam_recon': 1
}

training_parameters_imp = {
    'device_id': 0,
    'eval_step': 50,
    'k_folds': 5,
    'lr': 0.002,
    'num_epoch': 200,
    'rna_batchsize': 1024,
    'spatial_batchsize': 1024,
    'spatial_knn': 10
}

model_parameters_unm={
    'd_hidden': 64,
    'd_latent': 16,
    'lam_clf': 0.1,
    'lam_cos': 1,
    'lam_genegraph': 0,
    'lam_impute': 1,
    'lam_mmd': 0.1,
    'lam_recon': 1
}
training_parameters_unm={
    "device_id": 0,
    "k_folds": 5,
    "spatial_knn": 10
}

In [ ]:
celltype_df = pd.read_csv(f"{WORK_PATH}CSV/011_1st_clustering.csv", index_col="cell_id")

sp_list = []

for i, ss in enumerate(SAMPLES_SYMBOLS):
    sdata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{ss}")

    df = celltype_df[celltype_df.index.str.endswith(str(i))].copy()
    df.index = [x[:-1] + "1" for x in df.index]

    sp = sdata.tables["table"]
    sp = sp[sp.obs["cell_id"].isin(df.index)].copy()

    sp.obs = pd.merge(sp.obs[["cell_id"]], df, left_on="cell_id", right_index=True)

    # Extract all epithelial cells without subsampling
    sp = sp[sp.obs["cell_type"] == "Epithelial"].copy()

    sp.obs["sample"] = ss

    sp_list.append(sp)

    print(f"{ss}: {sp.n_obs:,} epithelial cells")

# Combine all epithelial cells from six samples
sp_all = ad.concat(sp_list, join="inner", label="sample", keys=list(SAMPLES_SYMBOLS), index_unique="-", merge="same")

print(f"\nTotal epithelial cells: {sp_all.n_obs:,}")
print(f"Total genes: {sp_all.n_vars:,}")
print(sp_all.obs["sample"].value_counts())


In [ ]:
test_genes = ["ABCA3", "AGER", "TP63", "FOXJ1"]

# Save ground truth
X_true = sp_all[:, test_genes].X

if sparse.issparse(X_true):
    X_true = X_true.toarray()
else:
    X_true = np.asarray(X_true)

true_df = pd.DataFrame(X_true, index=sp_all.obs_names, columns=test_genes)

sp_train = sp_all[:, ~sp_all.var_names.isin(test_genes)].copy()

sp_train.raw = None

assert not any(g in sp_train.var_names for g in test_genes)

# GSE135893 epithelial reference
ref_adata = sc.read_h5ad(f"{PUBLIC_PATH}GSE135893/ws/filtered_data.h5ad")

ref_epi = ref_adata[ref_adata.obs["population"] == "Epithelial"].copy()

assert set(test_genes).issubset(ref_epi.var_names)

save_dir = (f"{WORK_PATH}Imputation/stAI/validation_pooled_3genes")

# Train one model, holding out all three genes
stAI.main.run_impute(
    model_parameters=model_parameters_imp,
    training_parameters=training_parameters_imp,
    adata_spatial=sp_train.copy(),
    adata_rna=ref_epi.copy(),
    save_dir=save_dir
)

# Predict all three genes
res = stAI.main.impute_unmeasured_genes(
    genes_to_impute=test_genes,
    adata_spatial=sp_train.copy(),
    adata_rna=ref_epi.copy(),
    training_parameters=training_parameters_unm,
    model_parameters=model_parameters_unm,
    model_dir=save_dir,
    agg=True,
    device="cpu"
)

assert np.asarray(res).shape == (sp_train.n_obs, len(test_genes))

pred_df = pd.DataFrame(np.asarray(res), index=sp_train.obs_names, columns=test_genes)

pred_df.to_csv(f"{save_dir}/heldout_predictions.csv")
true_df.to_csv(f"{save_dir}/heldout_ground_truth.csv")


In [ ]:
def evaluate_subset(idx, sample_name):
    rows = []

    for gene in test_genes:
        y_true = true_df.loc[idx, gene].to_numpy()
        y_pred = pred_df.loc[idx, gene].to_numpy()

        valid = (
            np.isfinite(y_true) &
            np.isfinite(y_pred)
        )
        y_true = y_true[valid]
        y_pred = y_pred[valid]

        rho = np.nan
        r = np.nan

        if (
            len(y_true) >= 3
            and np.std(y_true) > 0
            and np.std(y_pred) > 0
        ):
            rho = spearmanr(y_true, y_pred).statistic
            r = pearsonr(y_true, y_pred).statistic

        rows.append({
            "sample": sample_name,
            "gene": gene,
            "n_cells": len(y_true),
            "detection_fraction": np.mean(y_true > 0),
            "spearman_rho": rho,
            "pearson_r": r
        })

    return rows

results = []

# All pooled cells
results.extend(evaluate_subset(sp_all.obs_names, "Pooled"))

# Each sample separately
for sample, group in sp_all.obs.groupby("sample", observed=True):
    results.extend(evaluate_subset(group.index, sample))

result_df = pd.DataFrame(results)

display(result_df)

result_df.to_csv(f"{save_dir}/validation_metrics.csv", index=False)

In [ ]:
df = result_df.copy()

sample_order = [
    "Pooled",
    "AF_s2",
    "FR_s1",
    "FR_s3",
    "FR_s4",
    "HC_s1",
    "HC_s3",
]

df["detection_pct"] = df["detection_fraction"] * 100

def make_matrix(column):
    return (
        df.pivot(index="sample", columns="gene", values=column)
        .reindex(index=sample_order, columns=test_genes)
    )

rho_mat = make_matrix("spearman_rho")
pearson_mat = make_matrix("pearson_r")
detect_mat = make_matrix("detection_pct")

n_cells = df.groupby("sample")["n_cells"].first().reindex(sample_order)
row_labels = [f"{s} (n={int(n_cells[s]):,})" for s in sample_order]

# Figure
fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))

panels = [
    (rho_mat, "Spearman correlation", "viridis", 0, 1, ".2f"),
    (pearson_mat, "Pearson correlation", "viridis", 0, 1, ".2f"),
    (detect_mat, "Detection fraction (%)", "Blues", 0, 100, ".0f"),
]

for i, (ax, panel) in enumerate(zip(axes, panels)):

    mat, title, cmap, vmin, vmax, fmt = panel
    values = mat.to_numpy(dtype=float)
    im = ax.imshow(
        values,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        interpolation="nearest"
    )
    ax.set_title(title, fontsize=12, pad=12)
    ax.set_xticks(np.arange(len(test_genes)))
    ax.set_xticklabels(
        gene_order,
        rotation=45,
        ha="right",
        va="top",
        fontsize=10,
        rotation_mode="anchor"
    )

    ax.set_yticks(np.arange(len(sample_order)))

    if i == 0:
        ax.set_yticklabels(row_labels, fontsize=10)
    else:
        ax.set_yticklabels([])

    ax.tick_params(length=0)

    for row in range(values.shape[0]):
        for col in range(values.shape[1]):

            value = values[row, col]

            label = f"{value:{fmt}}"

            if i == 2:
                label += "%"

            normalized = (value - vmin) / (vmax - vmin)

            text_color = "white" if normalized > 0.55 else "black"

            ax.text(col, row, label, ha="center", va="center", fontsize=10, color=text_color)
            
    ax.axhline(0.5, color="white", linewidth=2, )
    rec = Rectangle((-0.5, -0.5), len(test_genes), 1, fill=False, edgecolor="black", linewidth=2, zorder = 100)
    ax.add_patch(rec)

    ax.set_xlim(-0.6,3.55)
    ax.set_ylim(6.6, -0.6)

    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.025)
    cbar.ax.tick_params(labelsize=8)

fig.subplots_adjust(left=0.2, bottom=0.15, right=0.95, top=0.85)

fig.savefig(f"{WORK_PATH}Figure/012_stAI_heldout_validation_heatmap.png", dpi=300, format="png", bbox_inches=None)


In [ ]:
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/Epithelial.h5ad")

In [ ]:
pred_df.columns = [f"{c}_stai" for c in pred_df.columns]

In [ ]:
obs = pd.merge(sp_all.obs, pred_df, left_index=True, right_index=True).set_index("cell_id")

In [ ]:
adata.obs = obs

In [ ]:
vmax = {
    "ABCA3": [8, 2],
    "AGER":  [12, 3],
    "TP63":  [5, 1],
    "FOXJ1": [10, 2.2],
}

# Figure
fig, axes = plt.subplots(2, 4, figsize=(10, 4.5))
fig.subplots_adjust(left=0.05, right=0.95, top=0.92, bottom=0.07, wspace=0.25, hspace=0.2)
cmap = "viridis"

for i, gene in enumerate(test_genes):

    ax_t = axes[0, i]
    ax_p = axes[1, i]

    for ax, color, vmax_val in [
        (ax_t, gene, vmax[gene][0]),
        (ax_p, f"{gene}_stai", vmax[gene][1])
    ]:

        sc.pl.umap(
            adata, color=color, size=6, vmin=0, vmax=vmax_val,
            cmap=cmap, ax=ax, show=False, colorbar_loc=None, frameon=False, title=""
        )

        ax.set_axis_off()

        divider = make_axes_locatable(ax)

        cax = divider.append_axes("right", size="4%", pad=0.03)
        norm = mpl.colors.Normalize(vmin=0, vmax=vmax_val)
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        cb = fig.colorbar(sm, cax=cax)
        cb.ax.tick_params(labelsize=7, length=2, width=0.5)

        cb.outline.set_linewidth(0.5)

    ax_t.set_title(gene, fontsize=11, pad=7, style="italic")

fig.canvas.draw()

for row, label in enumerate(["Measured", "stAI-predicted"]):

    pos = axes[row, 0].get_position()
    y_center = (pos.y0 + pos.y1) / 2

    fig.text(0.035, y_center, label, rotation=90, ha="center", va="center", fontsize=11)

fig.savefig(f"{WORK_PATH}Figure/012_stAI_heldout_validation_umap.png", dpi=300, bbox_inches="tight", facecolor="white")

plt.show()